# CT–Report endpoint: register → evaluate/gate → deploy → invoke

End-to-end **CD** for the CT–report multimodal model on Azure ML (SDK v2), matching the
prototype architecture (`docs/13-mlops-and-cicd.md`):

1. **Register** the trained model from a training job as a `custom_model` **candidate** asset.
2. **Run the evaluate + gate pipeline** — it scores the candidate on a held-out shard and, if it
   clears the threshold (stand-in for the best current model), **registers the blessed model**
   `ct-report-siglip`.
3. **Deploy** the blessed model to a **managed online endpoint** (token auth, no keys).
4. **Invoke** the endpoint with sample CT volumes (images) and images+reports.

> Prototype on synthetic, de-identified data — not clinically validated. The scoring inputs are
> smoke-dimension volumes (32³) so payloads stay JSON-friendly.


## Environment (uv-managed, aligned kernel)

This notebook expects the **`CT-Report (uv)`** kernel, which is the project's uv virtual
environment — so the local imports (`ctreport`, `torch`) match the training/eval/serve code and
the Azure ML SDK v2 is available. From `prototype/model-training/ct-report-pretraining/`:

```bash
uv sync --extra notebook          # creates .venv with ctreport + torch + azure-ai-ml + jupyter
uv run python -m ipykernel install --user \
  --name ct-report-pretraining --display-name "CT-Report (uv)"
```

Then pick **Kernel → CT-Report (uv)**. (Add `--extra text` for the optional Qwen text tower.)


## 0. Connect

Uses `DefaultAzureCredential` (your `az login`). Fill in the governed workspace coordinates —
these match the IaC-provisioned workspace in `admin-setup/`.


In [ ]:
from azure.ai.ml import MLClient, Input
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment, CodeConfiguration, Environment, Model
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential

SUBSCRIPTION = "595a74d5-5d8a-421d-b364-979ba24a6489"
RESOURCE_GROUP = "EXP-ADO-ADB-RG"
WORKSPACE = "ctrp-mlw"

ml = MLClient(DefaultAzureCredential(), SUBSCRIPTION, RESOURCE_GROUP, WORKSPACE)
print("connected to", ml.workspace_name)

## 1. Register the candidate model from a training job

Point `TRAIN_JOB` at a completed training pipeline's Stage-2 child job. We register its `model`
output as a **v2 `custom_model` asset** `ct-report-siglip-candidate` — the input the evaluate
pipeline expects. (You can also do this with the CLI; see `scripts/deploy.md`.)


In [ ]:
# The Stage-2 child run name (…: get it from the training pipeline in the studio / `az ml job list`).
TRAIN_JOB = "REPLACE_WITH_STAGE2_JOB_NAME"

candidate = ml.models.create_or_update(Model(
    name="ct-report-siglip-candidate",
    path=f"azureml://jobs/{TRAIN_JOB}/outputs/model",
    type=AssetTypes.CUSTOM_MODEL,
    description="CT–report Stage-2 aligned model, candidate for the eval gate.",
))
print("registered", candidate.name, "v", candidate.version)

## 2. Run the evaluate + gate pipeline

Submit `azureml/evaluate-pipeline.yml`. On success the pipeline **registers the blessed model**
`ct-report-siglip` via its named output. If the gate fails, the job errors and nothing is
registered. (Threshold is permissive in the scaffold so the smoke happy-path demonstrates; in
production set it to the incumbent champion's score.)


In [ ]:
from azure.ai.ml import load_job

eval_job = load_job("../azureml/evaluate-pipeline.yml")
# pin the candidate we just registered
eval_job.inputs.candidate_model = Input(type=AssetTypes.CUSTOM_MODEL, path=f"{candidate.name}:{candidate.version}")
submitted = ml.jobs.create_or_update(eval_job)
print("eval pipeline:", submitted.name, submitted.studio_url)
ml.jobs.stream(submitted.name)   # blocks until done; raises if the gate fails

In [ ]:
blessed = ml.models.get("ct-report-siglip", label="latest")
print("blessed model:", blessed.name, "v", blessed.version)

## 3. Deploy the blessed model to a managed online endpoint

> ⚠️ **Prototype CD shortcut.** The steps below create the endpoint and shift 100% traffic
> **interactively with your Owner credentials** — no approval gate, no canary, no automated
> rollback, no service-principal isolation. Production CD embeds this in a pipeline / GitHub
> Actions run under a least-privilege workload identity with champion/challenger promotion and
> progressive delivery. See `scripts/deploy.md` and the README 'Standard MLOps' note — those
> concerns are intentionally out of scope for this prototype.

Create the endpoint (token auth) and a `blue` deployment from the local YAML. The first
environment image build can take ~15–25 min. `code_configuration` ships `src/` so the scoring
script's `from ctreport…` imports resolve alongside the weights-only model asset.


In [ ]:
endpoint = ManagedOnlineEndpoint(name="ct-report-endpoint", auth_mode="aml_token",
    description="Hosts the blessed CT–report multimodal model.")
ml.online_endpoints.begin_create_or_update(endpoint).result()

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name="ct-report-endpoint",
    model=f"{blessed.name}:{blessed.version}",
    code_configuration=CodeConfiguration(code="../src", scoring_script="score.py"),
    environment=Environment(conda_file="../environment/inference.yaml",
                            image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest"),
    instance_type="Standard_DS3_v2",
    instance_count=1,
)
ml.online_deployments.begin_create_or_update(deployment).result()

ep = ml.online_endpoints.get("ct-report-endpoint")
ep.traffic = {"blue": 100}
ml.online_endpoints.begin_create_or_update(ep).result()
print("endpoint ready; traffic:", ep.traffic)

## 4. Build sample requests (images, and images + reports)

Reuse the `ctreport` helpers so payloads match training: synthetic smoke-dim CT volumes with a
class-dependent HU shift + planted blob, and (optionally) the paired **report text**. Volumes
are flattened for JSON with an explicit `shape`.


In [ ]:
import sys, json, torch
sys.path.insert(0, "../src")
from ctreport.config import get_config
from ctreport.data.synthetic import SyntheticCTReportDataset, finding_class_names

cfg = get_config("smoke")
names = finding_class_names()
ds = SyntheticCTReportDataset(cfg, n_samples=3, seed=2024)   # unseen samples

def to_instance(sample, with_report):
    d, h, w = cfg.volume_shape
    inst = {"volume": sample["volume"].flatten().tolist(), "shape": [d, h, w]}
    if with_report:
        inst["report"] = sample["report"]   # raw report text; endpoint tokenizes it
    return inst

images_only = {"input_data": {"instances": [to_instance(ds[i], False) for i in range(3)]}}
images_text = {"input_data": {"instances": [to_instance(ds[i], True) for i in range(3)]}}
true_labels = [int(ds[i]["label"]) for i in range(3)]
print("true findings:", [names[l] for l in true_labels])

### 4a. Invoke with images only (zero-shot finding classification)

In [ ]:
import json, tempfile, os

def invoke(payload):
    with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False) as f:
        json.dump(payload, f); path = f.name
    try:
        raw = ml.online_endpoints.invoke(endpoint_name="ct-report-endpoint",
                                         deployment_name="blue", request_file=path)
    finally:
        os.remove(path)
    return json.loads(raw)

resp = invoke(images_only)
for i, p in enumerate(resp["predictions"]):
    print(f"[{i}] true={names[true_labels[i]]:32s} pred={p['predicted_finding']}")
resp

### 4b. Invoke with images + reports (adds image↔report cosine)

In [ ]:
resp2 = invoke(images_text)
for i, p in enumerate(resp2["predictions"]):
    print(f"[{i}] pred={p['predicted_finding']:32s} img-report cos={p.get('image_report_cosine'):.4f}")
resp2

## Notes

- **Auth:** endpoint is `aml_token` (no static keys) per governance; `invoke` uses your creds.
- **Text tower (Qwen vs fallback):** the model rebuilds the **pinned** text tower recorded in
  its `config.json` (`text_tower: qwen|fallback`). Requests send report **text** (`"report": "..."`)
  and the endpoint tokenizes it with that tower's tokenizer. A `qwen`-trained model needs
  `transformers`+`peft` and the Qwen tokenizer/config reachable offline (`CTREPORT_QWEN_DIR` +
  `HF_HUB_OFFLINE=1`) or `load_state_dict` fails — keep train and inference envs aligned. See
  `scripts/qwen-text-tower.md`. The `smoke` preset pins the dependency-free **fallback** tower.
- **Threshold (permissive on purpose):** the gate defaults to `0.20`, just above the 0.25
  random-chance floor, so a smoke model passes and this notebook runs end to end. It is **not** a
  quality bar — in production set it to the current champion model's metric on the same held-out
  set so only genuine improvements ship.
- **CD is not production-grade:** register/gate happen in the pipeline, but endpoint create +
  traffic here are manual under your Owner creds. Standard MLOps (CI-driven, service-principal,
  champion/challenger, canary + rollback, approvals + monitoring) is out of scope — see
  `scripts/deploy.md` and the README 'Standard MLOps' note.
- **Cleanup:** `ml.online_endpoints.begin_delete(name="ct-report-endpoint").result()` to avoid
  ongoing compute cost.
